# 02. Select four-month windows

For the SakAI final run, use the **25-day aligned timing recipe** with both adjacent-month rules. The lean export reuses the saved imagery audit, adds completed development downloads, and creates a new crop-stratified development split from the matched pool. The original 69-field benchmark remains in its saved run. Fixed windows use May through August.

**Start here for the final rerun.** Run 03 next to verify and review the feature handoff. A workspace without a saved feature baseline can first build its initial audit and feature handoff with the standard 01 → 02 → 03 workflow.

[Run order and reruns](../../docs/workflow.md).


In [1]:
from pathlib import Path
import sys
import json

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file())
sys.path.insert(0, str(root / "src"))
from crop_pipeline import lean_resplit, preprocessing_workflow as workflow
from crop_pipeline.notebook_display import show_table
from crop_pipeline.windows import seasons

baseline_features = ""  # Optional exact features.json from the original locked run.
quality_input = ""  # Set only for an explicit initial or historical quality-based run.


## Select and count qualifying windows

The final run permits adjacent-month threshold disagreement for the same supported cycle and gaps of up to 25 days between the readings that bracket green-up. Later confirmations still use the 20-day limit. The 80% imagery and eight-bin checks apply to each candidate.


In [ ]:
feature_pointer = workflow.pointer(root, "features")
saved_features = feature_pointer.is_file() and json.loads(feature_pointer.read_text()).get("status") == "prepared"
if baseline_features or (saved_features and not quality_input):
    feature_handoff = lean_resplit.run(root, baseline_features or None)
    window_handoff = feature_handoff.parent / "windows.json"
    show_table(seasons.read_table(feature_handoff.parent / "usable_samples.csv"),
               title="Usable samples for the final 25-day recipe")
    print("Feature handoff:", feature_handoff)
else:
    window_handoff = workflow.select(root, quality_input or None)
    show_table(seasons.read_table(window_handoff.parent / "window_counts.csv"),
               title="Complete windows before pixel capping")
print("Window handoff:", window_handoff)
print("Next: 03_feature_engineering.ipynb verifies this handoff.")


Detector outcomes: {'candidate': 3424, 'rejected': 4119, 'ambiguous': 2327}
Results: ./data/pipeline_work/analysis/external/external_t4tyh93t/greenup
Reference-supported field-seasons: 0
Initial estimated proposals: 2900
Automatically qualified estimates: 1894
Total qualified estimated windows: 1894
Unresolved outcomes: 7976
Automatic timing failures: 1006
Review templates: ./data/pipeline_work/analysis/external/external_t4tyh93t/seasons/review
Reference-supported windows saved: 0
Estimated observation windows saved: 1894
Audit: ./data/pipeline_work/analysis/external/external_t4tyh93t/seasons/season_review_audit.csv
Cycle selection: {'selected': 684, 'excluded_incomplete': 1124, 'excluded_later': 86}
Selected estimated windows: 684


## Evidence and interpretation

Coverage describes this saved snapshot. Green-up is estimated timing; sample targets do not establish accuracy. See [research basis and limits](../../docs/research_basis.md) for sources and project-specific choices.


## Figures

Parcel groups with a complete fixed or aligned window, followed by up to three saved green-up detection examples. A green-up is an algorithmic estimate, not a verified planting date.

In [ ]:
from IPython.display import Image, display
import matplotlib.pyplot as plt
from crop_pipeline.visualization import workflow_figures as figures


def show_and_save(figure, folder, name):
    """Display inline and keep a PNG beside the run outputs for reports and slides."""
    print("Saved figure:", figures.save_figure(figure, folder, name))
    display(figure)
    plt.close(figure)

figure_folder = window_handoff.parent / "figures"
window_counts = window_handoff.parent / "window_counts.csv"
if window_counts.is_file():
    show_and_save(figures.draw_window_counts(figures.read_table(window_counts)),
                  figure_folder, "window_counts")
greenup_examples = sorted((window_handoff.parent / "greenup" / "plots").glob("*.png"))[:3]
for example in greenup_examples:
    print("Green-up example:", example)
    display(Image(filename=str(example)))
if not greenup_examples:
    print("No saved green-up plots for this run.")